# Markets of Whiterun (Medieval Village) Agent-Based Simulation Models

Have you ever noticed how the towns and cities in fantasy and historical fiction never seem to have enough farms?

The goal of this project is to build an agent-based simulation model with decisions made by Cobb-Douglas marginal utility to find the number and behavior of farmers that lead to stable populations and stable commodity prices. We begin by running experiments meant to both find the range of parameters that lead to successful simulations, and establish the functionality of progressively more complex market mechanisms.
Once we have established a range of preference parameters that lead to stability, we will be able to derive world parameters from media and assess the resulting market dynamics--thus the titular markets of Whiterun.

The bulk of the code for this project will be done in the `.py` files, while results and summaries are saved for the `.ipynb` files.


In [3]:
# === IMPORT SHARED FUNCTIONS === #
import whiterunMarket as wrm  # import shared functions
import inspect                # allow us to see function source code in console


## World Parameters

The power of agent-based simulation models rests in its parameters. The terms listed below determine the basic rules by which our agents act.

For the sake of information gathering, we specify that each experiment runs 1,000 trials, `VILLAGES = int(1E3)`. Since our preference parameters are drawn from a random distribution, we need to ensure a large enough net is cast in each experiment. Each trial is run for 1,000 cycles, `DAYS = int(1E3)`, so as to allow for enough time to pass that unstable farmers will drop out of the sample. Because our agents don't interact with each other or with a dynamic market in the early experiments, the initial number of `FARMERS` in each trial is largely inconsequential; however, it will become one of the key parameters-of-interest for later experiments. For that reason, we set the count to be a dozen, so as to reflect a reasonable initial value for when that quantity begins to hold greater influence in the outcome of the simulation.

For the early experiments, the market and production parameters are somewhat arbitrary with the simplifying assumption that any solution produced within a reasonable range of values is congruent to a solution at a larger scale. Moreover, that once a baseline is established, specific scenarios can be assessed. As we develop the market mechanisms through iterative experiments, `PR_BREAD` and `PR_WHEAT` will become initial values for what will be dynamic, market-driven variables. Finally, once we have found that a stable market can be created, we can derive values for parameters like `FARMERS`, `PROD_WHEAT`, `EXHAUSTION`, etc. from fiction or real-world scenarios, and assess the results.


In [2]:
''' # Copied from whiterunMarket.py
# ['EXPERIMENT PARAMS']
VILLAGES   = int(1E3) # number of villages to text
DAYS       = int(1E3) # number of days to simulate

# ['WORLD PARAMS']
FARMERS    = 12  # number of farmer per village
PROD_WHEAT = 3 # amount of wheat produced per production action

# ['MARKET PARAMS']
PR_BREAD   = 3 # initial price of bread
PR_WHEAT   = 1 # initial price of wheat
'''

 # Copied from whiterunMarket.py
# ['EXPERIMENT PARAMS']
VILLAGES   = int(1E3) # number of villages to text
DAYS       = int(1E3) # number of days to simulate

# ['WORLD PARAMS']
FARMERS    = 12  # number of farmer per village
PROD_WHEAT = 3 # amount of wheat produced per production action

# ['MARKET PARAMS']
PR_BREAD   = 3 # initial price of bread
PR_WHEAT   = 1 # initial price of wheat



## Meet the Farmers

We define our farmer agent, or `agentFarmer(...)`, as having an inventory (`gold`, `wheat` and `bread`), a vector of preference params (`PREFS`), statuses (`hunger` and the living or dead binary `status`) and a utility function (the class method `.utility(...)`). Here we deploy the logarithmic-version of the Cobb-Douglas utility function for interpretability and to simulate diminishing returns of the resources.

Utility Function:
$U(G, W, B, H) = \gamma \ln(G+1) + \omega \ln(W+1) + \beta \ln(B+1) - \eta H$

Note that because none of our actions effect only one variable, taking the gradient will not be sufficient. Instead, we take the linear approach:
$\Delta U |_{\vec{x}} = U(X + \vec{x}) - U(X)$ where $X$ is the agent's current inventory and $\vec{x}$ is some change to the inventory.

Since we are taking a largely unsupervised (that is, little or no observed data) approach, the preference params will be drawn from some non-negative random distribution. The extent to which a set of preferences lead to a successful simulation will determine their explanatory power for our purposes--noting that falsifiability comes from the discussion of 

For now, we suppose a small number of Farmer Actions:
* Buy Bread : trades gold for bread
* Produce Wheat : creates wheat at the cost of additional hunger
* Sell Wheat : trades wheat for gold
* Idle : No change (marginal utility hardcoded to `0`)
  
In general, an agent will decide to take the action with the highest marginal utility ($\Delta U |_{\vec{x}}$). An agent cannot do an action they do not have the resources for (i.e. an agent cannot attempt to buy bread if the agent cannot afford bread). If all available actions produce negative marginal utility, then the agent will choose the `Idle` action. A $+1$ is included inside all logarithmic terms so that a `0` of that resource translates to `0` utility, instead of $-\infty$. If an agent's `hunger` exceeds some `STARVATION` threshold, then their `status` is set to `0`. This can reflect a farmer perishing, or could be adjust to also reflect bankruptcy. At the end of the day/ time-step, a farmer will consumer one bread if it is in inventory, reducing `hunger` to `0` or by some `amount`.


In [3]:
''' # Copied from whiterunMarket.py
# ['FARMER PARAMS']
# Farmer status change parameters
EXHAUSTION = 1 # amount of hunger gained per production action
STARVATION = 5 # max amount of hunger allowed before starvation
RECOVERY   = 0 # eating resets hunger if 0, else reduces hunger

# Initial conditions for farmer agent inventories
BREAD      = 1
WHEAT      = 0
GOLD       = 3
HUNGER     = 0'

# Use named tuple to make references easier
PARAMS = ['gamma', 'omega', 'beta', 'eta']
PREFS  = namedtuple('PREFS', PARAMS)
'''

print(
    '# Define a class for Farmer',
    inspect.getsource(wrm.agentFarmer),
    sep='\n'
)

 # Copied from whiterunMarket.py
    # ['FARMER PARAMS']
    # Farmer status change parameters
    EXHAUSTION = 1 # amount of hunger gained per production action
    STARVATION = 5 # max amount of hunger allowed before starvation
    RECOVERY   = 0 # eating resets hunger if 0, else reduces hunger

    # Initial conditions for farmer agent inventories
    BREAD      = 1
    WHEAT      = 0
    GOLD       = 3
    HUNGER     = 0

# Use named tuple to make references easier
PARAMS = ['gamma', 'omega', 'beta', 'eta']
PREFS  = namedtuple('PREFS', PARAMS)

# Define a class for Farmer
class agentFarmer :
    def __init__(self, func, **kwargs) :
        self.status = 1
        self.prefs  = PREFS(*func())
        self.bread  = kwargs.get('bread', BREAD)   # initial bread inventory
        self.wheat  = kwargs.get('wheat', WHEAT)   # initial wheat inventory
        self.gold   = kwargs.get('gold',  GOLD)    # initial gold inventory
        self.hunger = kwargs.get('hunger',HUNGER)  # initial hung

## Making Decisions on the Margin

Beginning in `experiment2`, agents gain the ability to make marginal transaction decisions--that is, decide whether to transact one more or one less rather than whether to transact or not. This is accomplished through an iterative process: the agent object is passed to the `agent_quantity(...)` function along with the price of the item and the agents relation to it in the transaction. Since the function is designed to be agent-agnostic and buyer-seller-agnostic, the latter 

In [4]:
print(
    inspect.getsource(wrm.agent_quantity)
)

def agent_quantity (
        agent, 
        price, 
        **kwargs
        ) :
    ''' Provided an agent-object with inventory and utility function, find the
        quantity of a good to transact until marginal utility is negative
    '''
    # Get resource relation -- 1 = increasing ; -1 = decreasing
    dWheat = kwargs.get('wheat', 0)
    dBread = kwargs.get('bread', 0)
    dGold  = kwargs.get('gold',  0)

    #   Define assistant functions
    # Create theoretical inventory, given (q)uantity transacted
    def state (q) :
        return dict(
            wheat = agent.wheat + dWheat*q,
            bread = agent.bread + dBread*q,
            gold  = agent.gold  + dGold*price*q
            )
    #   end state

    # Check if resources are exhausted
    def feasible (q) :
        s = state(q)
        return (s['wheat'] >= 0) and (s['bread'] >= 0) and (s['gold'] >= 0)
    #   end feasible

    quantity = 0  # create in memory
    prevUtil = agent.utility(**state(quantity)) 
    
   

## Coming soon...

* <b>Bakery & Mint :</b> how do our farmers react when there is no longer an unlimited supply of bread to purchase? What if wheat cannot be converted to bread to in the same time-step that a farmer sells it? We begin tracking the total amount of bread and gold in circulation in a village. This will mark the first time that a farmer's success is no longer independent of the other farmers.
* <b>Detour--Hunger-Interaction:</b> As it is, the `hunger` term in the utility functions seems to be under-appreciated in agent decision making, especially given its existential nature. How would behavior and survivability change if the utility function was altered to be...
*   - $U(G, W, B, H) = (\eta H + 1) \times (\alpha + \gamma \ln(G+1) + \omega \ln(W+1) + \beta \ln(B+1))$, or...
*   - $U(G, W, B, H) = (\eta H^2 + 1) \times (\alpha + \gamma \ln(G+1) + \omega \ln(W+1) + \beta \ln(B+1))$? 
      This would allow for `hunger` to exert a far greater influence in all decisions.
* <b>Bakers :</b> it's finally time to introduce agents on the other side of the transactions. Bakers (we could call them merchants, more generally) will be defined similarly to farmers, in that they will have an inventory and utility function. For the time being, they will not make regular action decisions (like farmers' produce wheat, sell wheat, etc.), instead they will always convert all of their wheat to bread at the start of the day and will always be available to transact with farmers. The crucial development here is that the quantity of goods a baker is willing to transact will be determined by their own marginal utility calculations--see `agent_quantity(...)` above. For this stage, we will continue to hold the prices of wheat and bread constant.
* <b>Detour--Evolutionary Farmers:</b> As it is, when a farmer's `hunger` exceed the `STARVATION` threshold, they are inactivated. The associated village could continue for several more time-steps or it could end there. Nevertheless, some information is gained and some information is lost. Taking inspiration from `MCMC`, when a farmer becomes inactive, we could replace it with a new farmer whose parameters are drawn from a distribution determined by the remaining farmers. In one sense this would be more efficient, in that villages no longer cease part way into their trials. It would also mark a considerable increase in the complexity of trying to draw robust inference from our posterior distributions--even under the most basic framework, we could no longer say that farmer success was independent.
* <b>The Marketplace :</b> building on the progress made by including bakers to the simulation, we will introduce the marketplace, allow for dynamic prices, and start tracking prices. Once farmers have made their action decision, those that have decided to transact will be passed to the market. Here we will take an iterative approach to processing transactions: 
*  - Starting with the final price from the previous time-step, calculate the total quantity of the good willing and able to be transacted at that price for buyers and sellers
*  - Select the lower of the two numbers to be the quantity transacted--this also determines which side has pricing power. Randomly assign that many transactions to buyers and sellers, taking care not to assign more transactions to an agent than they would want to transact.
*  - Once the quantity transacted has been exhausted, it is time to adjust the price. If buyers had the lower total quantity, reduce the price by one. If sellers had the lower total quantity, increase the price by one. Check to make sure there are still buyers and sellers will to transact at this new price. If so, restart with the new price. Otherwise, break the loop.
*  - Note that at this stage, we should consider combining the `sell wheat` and `buy bread` decisions into a single `go to market` decision. This would necessitate interlacing the market processes for wheat and bread so as to allow gold gained from selling a good to be spent in the same time-step. While it would represent a noted improvement in the marketplace simulation, it is also computationally intense.
* <b>Stochastic Crops:</b> When our farmer agents decide to harvest wheat, they are always able to produce a fixed number. While this is a helpful simplifying assumption, it may not be necessary to flatten the simulation in this way. It may render a more interesting portrayal of farming to have harvest success be determined by a draw from a non-negative, discrete random distribution. For now, the proposed function is a draw from a `Cauchy` distribution where the mean parameter increases on non-harvest days and decreases when harvested.
* <b>Detour--Decision sequencing:</b> We may find it necessary to pair <b>Stochastic Crops</b> with some change to the utility function that factors in memory of prior actions. The purpose of this being to dissuade repeated harvest actions.
* <b>Fix Currency in Circulation:</b> So far we have allowed bakers to cheat--creating gold out of thin air--to ensure that there was enough currency in circulation to allow as many mutually beneficial transactions as possible. Now that we have a sense of the range of gold necessary to create stable markets, it's time to fix the amount of currency in circulation. This is a crucial step in the development of the marketplace and has a high likely of creating snowballing crises. Too much currency will result in inflated prices; too little, and prices start to erode.
* <b>Detour--Money & Banking:</b> Even though developments in the understanding of currency and banking are some of the discoveries that mark the end of the Medieval era in real human history, it maybe beneficial to our simulation to introduce a Jarl agent type to play central banker and who derives utility from stable prices. This Jarl would have the ability to issue taxes, spending or stimulus to address instabilities or crises in the system. We would assume the Jarl had access to an effectively bottomless purse to spend from; however, the role of debt, private or public, has not been considered in the simulation yet.
* <b>The rest of the population:</b> Thus far, we have only consider two (possibly three) agent types: farmers and bakers (and the Jarl). While we should not feel the need to create an agent type for every role in society, since the goal of the project is to eventually devise a way to determine a relationship between number of farmers and the size of the population they support, creating a commoner (or everyone-else) class will eventually be necessary.
* <b>Whiterun:</b> Whiterun is a city in <i>Skyrim</i> (<i>The Elder Scrolls</i> series) and inspiration for this project. As such, this project cannot be considered concluded until we have derived the world and market parameters from <i>Skyrim</i>, assessed how well they hold up to our simulation, and then (assuming an insufficient number of farmers) finding how many farmers would be necessary to support Whiterun.
* ...dragons?